# FinGPT-Style Evaluation on Gold Commodity Dataset (Groq API Version)

This notebook mirrors the structure of your FinBERT notebook, but evaluates a **FinGPT-style sentiment workflow** on the Gold Commodity dataset using the **Groq API**.

It is set up so you can fill in your own key and model choice.


## 0. Installation

In [ ]:
%%capture
!pip install -U openai datasets scikit-learn pandas torch

## Device Check

In [ ]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")


## 1. Environment and Reproducibility Setup

In [ ]:
import random

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paste your Groq key directly below.
# Do not share this notebook or upload it publicly after adding your real key.
GROQ_API_KEY = "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"

print("GROQ_API_KEY set:", GROQ_API_KEY != "your_groq_api_key_here" and bool(GROQ_API_KEY))


## 2. Dataset Loading

In [ ]:
from datasets import load_dataset

gold_ds = load_dataset('SaguaroCapital/sentiment-analysis-in-commodity-market-gold')
split_name = 'test' if 'test' in gold_ds else 'train'
gold_raw_df = gold_ds[split_name].to_pandas()

sentence_col = next((c for c in ['sentence', 'text', 'content', 'headline'] if c in gold_raw_df.columns), None)
label_col = next((c for c in ['label_text', 'label', 'sentiment', 'target'] if c in gold_raw_df.columns), None)
if sentence_col is None or label_col is None:
    raise ValueError(f'Could not detect sentence/label columns in dataset columns={list(gold_raw_df.columns)}')

labels = gold_raw_df[label_col]
feature = gold_ds[split_name].features.get(label_col)
if hasattr(feature, 'names'):
    label_map = {i: name.lower() for i, name in enumerate(feature.names)}
    gold_raw_df['true_label'] = labels.map(lambda x: label_map.get(int(x), str(x).lower()) if pd.notnull(x) else x)
else:
    gold_raw_df['true_label'] = labels.astype(str).str.lower()

gold_raw_df = gold_raw_df[[sentence_col, 'true_label']].rename(columns={sentence_col: 'text'}).dropna()
gold_raw_df.head()


## 3. FinGPT-Style Model Setup via Groq

This notebook uses a **prompted chat-completion approach** instead of a classifier head, so it behaves more like a FinGPT-style instruction workflow.

Swap `MODEL_NAME` to any Groq-hosted model you want to test.


In [ ]:
import re
from openai import OpenAI

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("Replace the placeholder in GROQ_API_KEY with your actual Groq API key before running.")

# Replace this with the exact Groq-hosted model you want to test.
# Examples you can try if available on your account:
# MODEL_NAME = "llama-3.1-8b-instant"
# MODEL_NAME = "llama-3.3-70b-versatile"
# MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"
MODEL_NAME = "llama-3.1-8b-instant"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

print(f"Using Groq model: {MODEL_NAME}")


## 4. Prompt Template and Prediction Parser

In [ ]:
LABELS = ["negative", "neutral", "positive"]

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment of the given financial or commodity-market text. "
    "Return only one lowercase label from: negative, neutral, positive."
)

def build_prompt(text: str) -> str:
    return f"""Determine the sentiment of this financial text.

Text: {text}

Return only one label from: negative, neutral, positive."""

def parse_label(output_text: str) -> str:
    cleaned = output_text.strip().lower()

    for label in ["negative", "neutral", "positive"]:
        if re.search(rf'\b{label}\b', cleaned):
            return label

    if "bearish" in cleaned or "loss" in cleaned or "drop" in cleaned:
        return "negative"
    if "bullish" in cleaned or "gain" in cleaned or "rise" in cleaned:
        return "positive"

    return "neutral"


## 5. Batched Inference

In [ ]:
def groq_predict_one(text: str, temperature: float = 0) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(text)},
        ],
    )
    raw = response.choices[0].message.content.strip()
    return parse_label(raw)

def run_fingpt_batched(sentences, batch_size=8, temperature=0):
    # Groq chat completions are called one prompt at a time here.
    # batch_size is kept only to mirror the structure of your earlier notebook.
    preds = []
    total = len(sentences)

    for i, text in enumerate(sentences, start=1):
        pred = groq_predict_one(text, temperature=temperature)
        preds.append(pred)

        if i % batch_size == 0 or i == total:
            print(f"Processed {i}/{total}")

    return preds


## 6. Full Evaluation Metrics

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, matthews_corrcoef

true_labels = gold_raw_df["true_label"].tolist()
pred_labels = run_fingpt_batched(gold_raw_df["text"].tolist(), batch_size=20, temperature=0)

metrics = {
    "macro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0),
    "micro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="micro", zero_division=0),
    "weighted_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0),
    "accuracy": accuracy_score(true_labels, pred_labels),
    "mcc": matthews_corrcoef(true_labels, pred_labels),
}

metrics_df = pd.DataFrame([metrics])
print(metrics_df)

print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))


## 7. Save Predictions

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/fingpt_style_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pred_df = gold_raw_df.copy()
pred_df["pred_label"] = pred_labels
pred_df["model_name"] = MODEL_NAME
pred_df["provider"] = "groq"

csv_path = EXPORT_DIR / "gold_fingpt_style_predictions_groq.csv"
pred_df.to_csv(csv_path, index=False)

print(f"Saved predictions to: {csv_path}")


## 8. Optional Save of Run Metadata

Since this notebook uses API inference, there is no local model/tokenizer to save.
Instead, this cell saves the run configuration for reproducibility.


In [ ]:
from pathlib import Path
import json

MODEL_EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/fingpt_style_groq_run")
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run_config = {
    "provider": "groq",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "dataset": "SaguaroCapital/sentiment-analysis-in-commodity-market-gold",
    "labels": LABELS,
}

config_path = MODEL_EXPORT_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print(f"Saved run metadata to: {config_path}")


## 9. Cleanup

In [ ]:
import gc

for name in ["client"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Cleanup complete.")
